## Finite-rank construction for the eigen linear operator

This notebook verifies the finite-rank approximation using the functions `ϕ_i`, multipliers `μ_i`, and the right-hand side `rhs_i`.

For each index `i`, we construct the residual

$$
\mathrm{err}_i
=
\text{transport terms}
-
\text{linear terms}
-
\text{constraint terms}
+
\mathrm{rhs}_i,
$$

where

- transport terms are computed using `mul_A_gradB`,
- linear terms are applied via `apply(lin_mat, …)`,
- constraint terms involve `apply(I_mat, …)` and the multipliers,
- and `rhs_i` is supported only on β ∈ [0, Beta].

Because `rhs_i = 0` on β ∈ [Beta, L0], the residual is evaluated on the two regions separately and combined.

We assemble the Gram matrices

$$
GQ_{ij}
=
\frac{\langle \mathrm{rhs}_i, \mathrm{rhs}_j\rangle}{\lambda_i\,\lambda_j},
\qquad
GR_{ij}
=
\langle \mathrm{err}_i, \mathrm{err}_j\rangle,
$$

and compute

$$
\sqrt{\langle GQ, GR\rangle}
\quad\text{and}\quad
\sqrt{\langle GQ, GQ\rangle}.
$$

In [1]:
import Pkg
Pkg.activate("..")
include("../src/NS_Numerics.jl")
using .NS_Numerics

  Activating 

Using dtype = IntervalArithmetic

project at `~/Changhe/julia code/residual_verification`


.Interval{Float64}
Number of threads = 56


In [2]:
# Load the self-similar profile UP (frequency representation).
UP = from_mat_file("../data/UP.mat", 2; domain="frequency")

# Load the eigenfunction `up` and its eigenvalue `lbd`.
# This `up` is used in the orthogonality constraint through I(up).
up, lbd = from_mat_file("../data/up_eig.mat", 1; domain="frequency")
M0, M1 = size(UP["u3"].freq_func)

# Load RHS tensor: rhs_tensor[:,:,i,:] corresponds to rhs_i = Q ψ_i in physical space.
rhs_tensor = read_matrix("../data/rhs.mat", ["rhs_odd"])
K = size(rhs_tensor, 3)

# Load approximate eigenpairs defining the finite-rank basis.
eig_val, eig_vec = read_matrix("../data/eig.mat", ["eig_val_odd", "eig_vec_odd"])

# Basis-function bound parameters (Bessel zeros).
Y_paras, Z_paras = load_besselj_zeros("../data/bessel_zeros.mat"; refine = (dtype != Float64))

# Load approximate coefficient function Q (used in rhs = Q ψ).
Q = load_grad_U("../data/approx_Q.mat");

In [3]:
# Parameters for the β-domain splitting and norm estimates.
R = dt(32)
k = 12
a = UP.scl_fac / R
Beta = atan(inv(a)) # β split point determined by r↔β mapping
L1 = Pi / dt(2)

# W^{k,∞} bounds of eigenfunctions ψ_i (mapped into β-coordinates).
_, U_bound = lst_to_mat(Ur_bound_lst(Y_paras, Z_paras, k)...)
eig_bound_beta = r_to_beta(eig_fun_bound(U_bound, eig_vec), a);

# W^{k,∞} bound of Q (mapped into β-coordinates).
Q_Wk∞_mat = read_matrix("../data/Q_bound.mat", ["Q_WkInf"])
Q_Wk∞_mat_beta = r_to_beta(vec_to_mat(Q_Wk∞_mat), a)

# Bound for rhs_i = Q ψ_i using product/chain-rule based estimates.
# Inner products use weight tanβ/cosβ, so this factor must be included 
# when computing W^{k,∞} bounds for rhs.
B  = tan_over_cos_deriv_Linf_bounds(Beta * dt("1.01"), k)
rhs_beta = multinomial_sum(k, B, Q_Wk∞_mat_beta, eig_bound_beta; last_flag=false);

In [4]:
# Assemble unknowns ϕ_i and constraint multipliers μ_i, together with rhs_i = Q ψ_i.
up_phi_lst = []
rhs_lst = []
rhs_norms_lst = []
lbd_lst = []
for ind in 1:K
    println("i = $ind / $K")

    # Load ϕ_i, μ_i (Lagrangian multiplier for the orthogonality condition)
    up_phi, lbd_phi = from_mat_file("../data/phi_odd/up_phi_$(ind-1).mat", 1; domain="frequency")
    push!(up_phi_lst, up_phi)
    push!(lbd_lst, lbd_phi)

    # rhs_i in physical-space VectorFunc.
    rhs = VectorFunc(UP.scl_fac, "space"; e1=rhs_tensor[:, :, ind, 1], e2=rhs_tensor[:, :, ind, 2], e3=rhs_tensor[:, :, ind, 3])
    push!(rhs_lst, rhs)
    
    # W^{k-2,∞} bound for rhs_i (used in inner_product estimates later).
    paras = Linf_to_paras(rhs_beta, k, [0, dt(410)]; ind=ind)
    rhs_norms = norm_lst(rhs, "W$(k-2)Inf"; paras=paras, domain=[Beta, L1])
    push!(rhs_norms_lst, rhs_norms)
end

i = 1 / 24
i = 2 / 24
i = 3 / 24
i = 4 / 24
i = 5 / 24
i = 6 / 24
i = 7 / 24
i = 8 / 24
i = 9 / 24
i = 10 / 24
i = 11 / 24
i = 12 / 24
i = 13 / 24
i = 14 / 24
i = 15 / 24
i = 16 / 24
i = 17 / 24
i = 18 / 24
i = 19 / 24
i = 20 / 24
i = 21 / 24
i = 22 / 24
i = 23 / 24
i = 24 / 24


In [5]:
# Use a full β-grid to obtain rigorous W^{k,∞} bounds for operator-applied terms.
L0 = Pi * dt(2)

# Build the β and θ points as column‐vectors
N0, N1 = 16000, 2000
β_pt = dt.(0:N0) .*(L0/dt(N0))
θ_pt = dt.(0:N1) .*(L1/dt(N1))

# Operator matrices:
# - lin_mat: base linear part of L
# - I_mat:  constraint-direction operator I(·) used with multipliers μ_i
lin_mat = get_operator_matrix(up, β_pt, θ_pt, "linear")
I_mat = get_operator_matrix(up, β_pt, θ_pt, "identity");


In [6]:
# Precompute operator-side terms and their W^{k,∞} bounds.
# Implemented operator-side combination (per i):
#   lfs_i = [UP⋅∇ϕ_i + ϕ_i⋅∇UP]  -  L_base(ϕ_i)  -  I(ϕ_i)*lbd  -  I(up)*μ_i
# (Signs follow the notebook implementation.)
lfs_lst = vec(mul_A_gradB(UP, up_phi_lst, β_pt, θ_pt)) .+ vec(mul_A_gradB(up_phi_lst, UP, β_pt, θ_pt))
lfs_Wk∞_norm_lst = []
UP_Wk∞_lst = []

freq_lst = dt.([4 * M0 + 10, 4 * M1 + 10])
UP_freq_lst = dt.([2 * M0 + 10, 2 * M1 + 10])

# I(up) and its W^{k-2,∞} bound (used in constraint terms).
I_up = apply(I_mat, up)
up_Wk∞_norm = norm_lst(I_up, "W$(k-2)Inf"; paras=UP_freq_lst, domain=[L0,L1])
for ind in 1:K
    println("i = $ind / $K")
    
    # I(ϕ_i) and its W^{k-2,∞} bound.
    I_up_phi = apply(I_mat, up_phi_lst[ind])
    UP_Wk∞_norm = norm_lst(I_up_phi, "W$(k-2)Inf"; paras=UP_freq_lst, domain=[L0,L1])
    push!(UP_Wk∞_lst, UP_Wk∞_norm)

    # Operator-side combination lfs_i.
    lfs = lfs_lst[ind] - apply(lin_mat, up_phi_lst[ind]) - I_up_phi * lbd - I_up * lbd_lst[ind]

    # W^{k,∞} bound for lfs_i (used later for residual bounds in the rhs=0 region).
    lfs_Wk∞_norm = norm_lst(lfs, "W$(k)Inf"; paras=freq_lst, domain=[L0,L1])
    push!(lfs_Wk∞_norm_lst, lfs_Wk∞_norm)
end

i = 1 / 24
i = 2 / 24
i = 3 / 24
i = 4 / 24
i = 5 / 24
i = 6 / 24
i = 7 / 24
i = 8 / 24
i = 9 / 24
i = 10 / 24
i = 11 / 24
i = 12 / 24
i = 13 / 24
i = 14 / 24
i = 15 / 24
i = 16 / 24
i = 17 / 24
i = 18 / 24
i = 19 / 24
i = 20 / 24
i = 21 / 24
i = 22 / 24
i = 23 / 24
i = 24 / 24


In [7]:
# Free large arrays no longer needed.
rhs_tensor = nothing
lfs_lst = nothing
GC.gc()

In [8]:
# Refine the grid on the rhs-support region β ∈ [0, Beta].
# We pad by (k-2) points on both ends so that after trimming,
# the interior region aligns with the derivative-based (W^{k-2,∞}) estimates.
N0 = 12000
N1 = 6000

# Extend the grids for finite difference method
β_pt = dt.(-k+2:N0+k-2) .*(Beta/dt(N0))
θ_pt = dt.(-k+2:N1+k-2) .*(L1/dt(N1))

# Generate the operators mat to compute the residual
lin_mat = get_operator_matrix(up, β_pt, θ_pt, "linear")
I_mat = get_operator_matrix(up, β_pt, θ_pt, "identity");

In [9]:
# Residual on β ∈ [0, Beta] (rhs support region).
# Start with transport terms, then add rhs and subtract (linear + constraint) terms.
err_lst = vec(mul_A_gradB(UP, up_phi_lst, β_pt, θ_pt)) .+ vec(mul_A_gradB(up_phi_lst, UP, β_pt, θ_pt))

I_up = apply(I_mat, up)
err_norms_lst = []
for ind in 1:K
    println("i = $ind / $K")
    
    err = err_lst[ind]

    # linear + constraint terms on the LHS
    lfs = apply(lin_mat, up_phi_lst[ind]) + apply(I_mat, up_phi_lst[ind]) * lbd + I_up * lbd_lst[ind]

    # err = transport + rhs - (linear + constraint)
    add_equal!(err, rhs_lst[ind] - lfs)

    # W^{k-2,∞} bound for err, combining rhs bound and operator-side bound.
    err_norm = [rhs_beta[i][:, ind] + lfs_Wk∞_norm_lst[ind][i,1] for i in 1: k+1]
    paras = Linf_to_paras(err_norm, k, [0, dt(610)]; ind=1)
    err_norms = norm_lst(err, "W$(k-2)Inf"; paras=paras, domain=[Beta, L1])
    push!(err_norms_lst, err_norms)

    # Remove the padded (k-2) boundary layers: keep only the interior points.
    trim_border!(err, k-2)
    trim_border!(rhs_lst[ind], k-2)
end

i = 1 / 24
i = 2 / 24
i = 3 / 24
i = 4 / 24
i = 5 / 24
i = 6 / 24
i = 7 / 24
i = 8 / 24
i = 9 / 24
i = 10 / 24
i = 11 / 24
i = 12 / 24
i = 13 / 24
i = 14 / 24
i = 15 / 24
i = 16 / 24
i = 17 / 24
i = 18 / 24
i = 19 / 24
i = 20 / 24
i = 21 / 24
i = 22 / 24
i = 23 / 24
i = 24 / 24


In [10]:
# Assemble Gram matrices (rhs support region):
# - GQ: rhs Gram matrix scaled by eigenvalues
# - GR: residual Gram matrix
# Detailed explanation of residual splitting and Gram matrix assembly is given in
# selfsimilar_linear_operator_finite_rank.ipynb (see the markdown cells there).
GQ = zeros(dtype, K, K)
GR = zeros(dtype, K, K)

for i in 1:K, j in i:K
    println("i = $i / $K, j = $j / $K")
    GQ[i, j] = inner_product(rhs_lst[i], rhs_lst[j]; W_norms_lst=[rhs_norms_lst[i], 
        rhs_norms_lst[j]], domain=[Beta, L1]) / (eig_val[j] * eig_val[i])

    GR[i, j] = inner_product(err_lst[i], err_lst[j]; W_norms_lst=[err_norms_lst[i], 
        err_norms_lst[j]], domain=[Beta, L1])
    
    # Gram matrices are symmetric: fill the lower triangle.
    if i != j
        GQ[j, i] = GQ[i, j]
        GR[j, i] = GR[i, j]
    end
end

i = 1 / 24, j = 1 / 24
i = 1 / 24, j = 2 / 24
i = 1 / 24, j = 3 / 24
i = 1 / 24, j = 4 / 24
i = 1 / 24, j = 5 / 24
i = 1 / 24, j = 6 / 24
i = 1 / 24, j = 7 / 24
i = 1 / 24, j = 8 / 24
i = 1 / 24, j = 9 / 24
i = 1 / 24, j = 10 / 24
i = 1 / 24, j = 11 / 24
i = 1 / 24, j = 12 / 24
i = 1 / 24, j = 13 / 24
i = 1 / 24, j = 14 / 24
i = 1 / 24, j = 15 / 24
i = 1 / 24, j = 16 / 24
i = 1 / 24, j = 17 / 24
i = 1 / 24, j = 18 / 24
i = 1 / 24, j = 19 / 24
i = 1 / 24, j = 20 / 24
i = 1 / 24, j = 21 / 24
i = 1 / 24, j = 22 / 24
i = 1 / 24, j = 23 / 24
i = 1 / 24, j = 24 / 24
i = 2 / 24, j = 2 / 24
i = 2 / 24, j = 3 / 24
i = 2 / 24, j = 4 / 24
i = 2 / 24, j = 5 / 24
i = 2 / 24, j = 6 / 24
i = 2 / 24, j = 7 / 24
i = 2 / 24, j = 8 / 24
i = 2 / 24, j = 9 / 24
i = 2 / 24, j = 10 / 24
i = 2 / 24, j = 11 / 24
i = 2 / 24, j = 12 / 24
i = 2 / 24, j = 13 / 24
i = 2 / 24, j = 14 / 24
i = 2 / 24, j = 15 / 24
i = 2 / 24, j = 16 / 24
i = 2 / 24, j = 17 / 24
i = 2 / 24, j = 18 / 24
i = 2 / 24, j = 19 / 24
i = 2 / 2

In [11]:
# Complementary region β ∈ [Beta, L0], padded by (k-2) points for trimming.
# In this region, rhs vanishes, so residual contains only operator-side terms.

L0 = Pi / dt(2)
N0 = 6000
N1 = 6000
β_pt = dt.(-k+2:N0+k-2) .*((L0 - Beta)/dt(N0)) .+ Beta
θ_pt = dt.(-k+2:N1+k-2) .*(L1/dt(N1))

lin_mat_phi = get_operator_matrix(up, β_pt, θ_pt, "linear")
I_mat = get_operator_matrix(up, β_pt, θ_pt, "identity");

In [12]:
# Residual on β ∈ [Beta, L0] where rhs = 0:
# err = transport - (linear + constraint)
err_lst = vec(mul_A_gradB(UP, up_phi_lst, β_pt, θ_pt)) .+ vec(mul_A_gradB(up_phi_lst, UP, β_pt, θ_pt))
I_up = apply(I_mat, up)
err_norms_lst2 = []

for ind in 1:K
    println("i = $ind / $K")

    # rhs = 0 here, so err = transport - lfs
    err = err_lst[ind]
    lfs = apply(lin_mat_phi, up_phi_lst[ind]) + apply(I_mat, up_phi_lst[ind]) * lbd + I_up * lbd_lst[ind]
    minus_equal!(err, lfs)

    # W^{k-2,∞} bound for this region's residual (operator-side bound only).
    err_norm = lfs_Wk∞_norm_lst[ind][:,1]
    paras = Linf_to_paras(err_norm, k, [0, dt(610)]; ind=1)
    err_norms = norm_lst(err, "W$(k-2)Inf"; paras=paras, domain=[L0-Beta, L1])
    push!(err_norms_lst2, err_norms)

    trim_border!(err, k-2)
end

i = 1 / 24
i = 2 / 24
i = 3 / 24
i = 4 / 24
i = 5 / 24
i = 6 / 24
i = 7 / 24
i = 8 / 24
i = 9 / 24
i = 10 / 24
i = 11 / 24
i = 12 / 24
i = 13 / 24
i = 14 / 24
i = 15 / 24
i = 16 / 24
i = 17 / 24
i = 18 / 24
i = 19 / 24
i = 20 / 24
i = 21 / 24
i = 22 / 24
i = 23 / 24
i = 24 / 24


In [13]:
# Accumulate GR contribution from β ∈ [Beta, L0] (rhs=0 region).
# Use the region-2 residual bounds err_norms_lst2.

for i in 1:K, j in i:K
    println("i = $i / $K, j = $j / $K")
    GR[i, j] += inner_product(err_lst[i], err_lst[j]; W_norms_lst=[err_norms_lst2[i], 
        err_norms_lst2[j]], domain=[L0-Beta, L1])

    # Symmetry fill
    if i != j
        GR[j, i] = GR[i, j]
    end
end

i = 1 / 24, j = 1 / 24
i = 1 / 24, j = 2 / 24
i = 1 / 24, j = 3 / 24
i = 1 / 24, j = 4 / 24
i = 1 / 24, j = 5 / 24
i = 1 / 24, j = 6 / 24
i = 1 / 24, j = 7 / 24
i = 1 / 24, j = 8 / 24
i = 1 / 24, j = 9 / 24
i = 1 / 24, j = 10 / 24
i = 1 / 24, j = 11 / 24
i = 1 / 24, j = 12 / 24
i = 1 / 24, j = 13 / 24
i = 1 / 24, j = 14 / 24
i = 1 / 24, j = 15 / 24
i = 1 / 24, j = 16 / 24
i = 1 / 24, j = 17 / 24
i = 1 / 24, j = 18 / 24
i = 1 / 24, j = 19 / 24
i = 1 / 24, j = 20 / 24
i = 1 / 24, j = 21 / 24
i = 1 / 24, j = 22 / 24
i = 1 / 24, j = 23 / 24
i = 1 / 24, j = 24 / 24
i = 2 / 24, j = 2 / 24
i = 2 / 24, j = 3 / 24
i = 2 / 24, j = 4 / 24
i = 2 / 24, j = 5 / 24
i = 2 / 24, j = 6 / 24
i = 2 / 24, j = 7 / 24
i = 2 / 24, j = 8 / 24
i = 2 / 24, j = 9 / 24
i = 2 / 24, j = 10 / 24
i = 2 / 24, j = 11 / 24
i = 2 / 24, j = 12 / 24
i = 2 / 24, j = 13 / 24
i = 2 / 24, j = 14 / 24
i = 2 / 24, j = 15 / 24
i = 2 / 24, j = 16 / 24
i = 2 / 24, j = 17 / 24
i = 2 / 24, j = 18 / 24
i = 2 / 24, j = 19 / 24
i = 2 / 2

In [14]:
# Full-domain grid (no padding) for L2 norms / inner products of I(up) and I(ϕ_i).
N0 = 12000
N1 = 6000
β_pt = dt.(0:N0) .*(L0/dt(N0))
θ_pt = dt.(0:N1) .*(L1/dt(N1))

I_mat = get_operator_matrix(up, β_pt, θ_pt, "identity");

In [15]:
# Compute L2 norms of I(up), I(ϕ_i) and their inner products <I(up), I(ϕ_i)>.
# These quantities are later used to bound the effect of uncertainty in up (interval arithmetic).

I_up = apply(I_mat, up)
up_norm = norm_lst(I_up, "L2"; paras=up_Wk∞_norm, domain=[L0,L1])
up_phi_norm_lst = []
up_phi_inpd_lst = []

for ind in 1:K
    println("i = $ind / $K")
    I_up_phi = apply(I_mat, up_phi_lst[ind])
    push!(up_phi_inpd_lst, inner_product(I_up, I_up_phi; W_norms_lst=[up_Wk∞_norm, UP_Wk∞_lst[ind]], domain=[L0, L1]))
    push!(up_phi_norm_lst, norm_lst(I_up_phi, "L2"; paras=UP_Wk∞_lst[ind], domain=[L0,L1]))
end

i = 1 / 24
i = 2 / 24
i = 3 / 24
i = 4 / 24
i = 5 / 24
i = 6 / 24
i = 7 / 24
i = 8 / 24
i = 9 / 24
i = 10 / 24
i = 11 / 24
i = 12 / 24
i = 13 / 24
i = 14 / 24
i = 15 / 24
i = 16 / 24
i = 17 / 24
i = 18 / 24
i = 19 / 24
i = 20 / 24
i = 21 / 24
i = 22 / 24
i = 23 / 24
i = 24 / 24


## Derivation of the correction term

The functions φ_i are constructed using a constraint term in the direction I(up).
This introduces a component in the residual proportional to

$$
c_i I(up),
$$

where the coefficient is the projection

$$
c_i =
\frac{
\langle I(up), I(\phi_i) \rangle
}{
\|I(up)\|^2
}.
$$

If up were an exact kernel of the linear operator L, then L(up) would be zero.
However, we only know that

$$
\|L(up)\| \le up\_err\_norm\_bd.
$$

By linearity of the operator,

$$
L(c_i\,up) = c_i\,L(up),
$$

so the induced residual component satisfies

$$
\|\delta_i\|
=
\|c_i\,L(up)\|
\le
|c_i| \cdot up\_err\_norm\_bd.
$$

Define

$$
a_i = |c_i| \cdot up\_err\_norm\_bd.
$$

If the true residual is

$$
\tilde e_i = e_i + \delta_i,
$$

then

$$
\langle \tilde e_i, \tilde e_j \rangle
=
\langle e_i, e_j \rangle
+
\langle \delta_i, e_j \rangle
+
\langle e_i, \delta_j \rangle
+
\langle \delta_i, \delta_j \rangle.
$$

Using

$$
\|\delta_i\| \le a_i
\quad\text{and}\quad
\|e_i\| = \sqrt{GR_{ii}},
$$

we obtain

$$
|\Delta GR_{ij}|
\le
a_i a_j
+
a_i \sqrt{GR_{jj}}
+
a_j \sqrt{GR_{ii}}.
$$

This is exactly the correction added to GR in the code.

In [16]:
# If using interval arithmetic, add a conservative correction to GR
# to account for uncertainty in up projected through the constraint direction.

if dtype != Float64
    up_err_norm_bd = dt("2e-5")
    GR_cor_lst = [abs(inpd) / up_norm^2 * up_err_norm_bd for inpd in up_phi_inpd_lst]
    GR_cor_mat = [GR_cor_lst[i] * GR_cor_lst[j] + GR_cor_lst[i] * sqrt(GR[j, j]) + GR_cor_lst[j] * sqrt(GR[i, i]) 
        for i in 1:K, j in 1:K]
    GR = add_error(GR, GR_cor_mat)
end;

In [ ]:
# Frobenius-type matrix inner products.
sum_err = sum(GR[i, j] * GQ[i, j] for i in 1:K, j in 1:K)
println("sqrt(<GQ, GR>): ", sqrt(sum_err))

sum_err = sum(GQ[i, j] * GQ[i, j] for i in 1:K, j in 1:K)
println("sqrt(<GQ, GQ>): ", sqrt(sum_err))

# Aggregate size of {I(ϕ_i)} in L2:
# - sum_sq = Σ ||I(ϕ_i)||_{L2}^2
# - sqrt_sum_sq = sqrt(sum_sq)
sum_norm = norm(up_phi_norm_lst)
println("\\sqrt(\\sum_{i} || ϕ_i ||_{L^2}^2) = ", sum_norm)

sqrt(<GQ, GR>): [0.0, 0.0039055]_trv
sqrt(<GQ, GQ>): [0.0259178, 0.0281468]_trv
\sum_{i} || ϕ_i ||_{L^2}^2 = [1.01343, 1.01343]_trv
